In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import json
from datetime import datetime
import pandas as pd
import time

In [2]:
def grid_parser(url):
    # url = "https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-16"

    options = Options()
    options.page_load_strategy = "eager"
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(1)

    rows = []
    iframes = driver.find_elements(By.CSS_SELECTOR, "iframe[src*='crosshare']")

    for idx in range(len(iframes)):
        iframes = driver.find_elements(By.CSS_SELECTOR, "iframe[src*='crosshare']")
        iframe = iframes[idx]
        src = iframe.get_attribute("src")
        print("src:", src)
        if not src or "crosshare" not in src:
            continue

        driver.switch_to.frame(iframe)
        print("a")
        time.sleep(0.5)

        script = driver.find_element(By.ID, "__NEXT_DATA__")
        json_text = script.get_attribute("innerHTML")

        data = json.loads(json_text)
        puzzle = data["props"]["pageProps"]["puzzle"]
        grid = puzzle["grid"]
        cols = puzzle["size"]["cols"]
        t = puzzle["publishTime"]
        dt = datetime.fromtimestamp(t / 1000).date()

        words = []

        # across words
        for i in range(0, len(grid), cols):
            row = grid[i:i+cols]
            row_str = "".join(row)

            for word in row_str.split("."):
                if len(word) >= 2:
                    words.append(word)

        # down words
        for c in range(cols):
            col_chars = []
            for r in range(0, len(grid), cols):
                col_chars.append(grid[r + c])

            col_str = "".join(col_chars)

            for word in col_str.split("."):
                if len(word) >= 2:
                    words.append(word)

        for w in words:
            rows.append({"Date": dt, "Word": w})

        driver.switch_to.default_content()

    df = pd.DataFrame(rows)
    driver.quit()
    return df

In [ ]:
def retrieveArticles(page, page_size):
    baseurl = f"https://www.cavalierdaily.com/section/quizzes?page={page}&per_page={page_size}"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(baseurl, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href = True):
        href = a["href"] 
        print(href)
        if "/crossword" in href.lower():
            if href not in links:
                links.append(href)
    return links


In [6]:
df = pd.DataFrame()
links = retrieveArticles(1,200)
print(links)
for l in links:
    print("link", l)
    articledf = grid_parser(l)
    df = pd.concat([df, articledf], ignore_index=True)

https://twitter.com/cavalierdaily/
http://www.facebook.com/CavalierDaily/
https://instagram.com/cavalierdaily/
https://open.spotify.com/show/72aDKLUZHKgbyrcfr1gITs?si=ZhBQU6LDTiu6snR6GsIJmA/
http://www.youtube.com/user/CavalierDaily/
http://eepurl.com/h4p1zP
https://www.cavalierdaily.com/page/donate
https://www.cavalierdaily.com/section/spanish
https://www.cavalierdaily.com/section/chinese
https://www.cavalierdaily.com/
https://www.cavalierdaily.com/section/news
https://www.cavalierdaily.com/section/news
https://www.cavalierdaily.com/section/election-2025
https://www.cavalierdaily.com/section/student-life
https://www.cavalierdaily.com/section/administration
https://www.cavalierdaily.com/section/city-government
https://www.cavalierdaily.com/section/self-governance
https://www.cavalierdaily.com/section/local-news
https://www.cavalierdaily.com/section/focus
https://www.cavalierdaily.com/section/sports
https://www.cavalierdaily.com/section/sports
https://www.cavalierdaily.com/section/sport

KeyboardInterrupt: 